# Lesson 1: Python Thinks Like a Hacker

**Welcome!** In this lesson, we will learn the basics of Python programming by using it as a cybersecurity tool. instead of just printing "Hello World", we will learn how hackers and security professionals view the web.

## 1. Introduction: The Attacker vs. Defender Mindset
Cybersecurity is not just about tools; it's about a mindset. 
- **Attackers** look for **one** weak point to enter.
- **Defenders** must secure **all** points.

**Ethics:** The skills you learn here are powerful. Unauthorized testing of systems you do not own is illegal and unethical. Always have permission before scanning or testing any website. We will use safe, educational targets today.

## 2. Python Fundamentals: Variables & Strings
In Python, we store data in **variables**. Text is called a **String**.

Let's look at a simple example where we store a URL.

In [1]:
# Creating a variable
target_url = "https://example.com"

# Printing the variable
print("Targetting:", target_url)

Targetting: https://example.com


### Lists and Loops
Often, we want to work with multiple things at once (like a list of passwords or URLs). We use **Lists** for storage and **Loops** to go through them.

In [2]:
# A list of common admin paths
admin_paths = ["/admin", "/login", "/dashboard", "/config"]

# Looping through the list
for path in admin_paths:
    full_url = target_url + path
    print("Checking potential path:", full_url)

Checking potential path: https://example.com/admin
Checking potential path: https://example.com/login
Checking potential path: https://example.com/dashboard
Checking potential path: https://example.com/config


## 3. The `requests` Library
Browsers like Chrome or Firefox send **HTTP Requests** to servers to get web pages. Python can do this too using the `requests` library. This is how scripts interact with the web.

In [3]:
import requests
import urllib3

# Suppress the warning that we are making an unverified HTTPS request
# In production, you would fix the certificate issue instead of suppressing it.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Send a GET request to the target
# verify=False is often used in security testing to bypass SSL errors from proxies or self-signed certs
response = requests.get(target_url, verify=False)

# Check the status code (200 means OK)
print("Status Code:", response.status_code)

# Limit output to first 500 characters so we don't spam the screen
print("Page Content Preview:\n", response.text[:500])

Status Code: 200
Page Content Preview:
 <!doctype html><html lang="en"><head><title>Example Domain</title><meta name="viewport" content="width=device-width, initial-scale=1"><style>body{background:#eee;width:60vw;margin:15vh auto;font-family:system-ui,sans-serif}h1{font-size:1.5em}div{opacity:0.8}a:link,a:visited{color:#348}</style></head><body><div><h1>Example Domain</h1><p>This domain is for use in documentation examples without needing permission. Avoid use in operations.</p><p><a href="https://iana.org/domains/example">Learn more<


/Users/yuliiayermolova/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 4. Reading HTTP Headers
When a server responds, it sends **Headers** (metadata) before the actual HTML content. These headers can tell us a lot about the security of the server.

Common Security Headers we look for:
- `X-Frame-Options`: Prevents Clickjacking.
- `Strict-Transport-Security` (HSTS): Enforces HTTPS.
- `Content-Security-Policy` (CSP): Prevents XSS.

Let's inspect the headers of our target.

In [4]:
# Access headers from the response object
headers = response.headers

# Print all headers nicely
for key, value in headers.items():
    print(f"{key}: {value}")

print("-" * 30)

# Check for a specific security header
security_header = "X-Frame-Options"

if security_header in headers:
    print(f"[+] Found {security_header}: {headers[security_header]}")
else:
    print(f"[-] Missing {security_header} - Potential vulnerability!")

Date: Wed, 22 Apr 2026 16:25:17 GMT
Content-Type: text/html
Transfer-Encoding: chunked
Connection: keep-alive
Server: cloudflare
last-modified: Sat, 18 Apr 2026 00:49:31 GMT
allow: GET, HEAD
Age: 10345
cf-cache-status: HIT
Content-Encoding: gzip
CF-RAY: 9f05fbca7e5e56ab-OSL
------------------------------
[-] Missing X-Frame-Options - Potential vulnerability!


## 5. Introduction to XSS (Cross-Site Scripting)

**XSS** occurs when an attacker can inject malicious code (usually JavaScript) into a website that other users view.

The most basic sign of XSS vulnerability is when a site reflects user input without sanitizing it. We often look for the `<script>` tag as a proof-of-concept.

Let's try to find a `<script>` tag in a piece of HTML code.

In [5]:
# Simulated HTML content (imagine this came from a website)
html_content = """
<html>
  <body>
    <h1>Welcome!</h1>
    <p>User comment: <script>alert('Hacked!');</script></p>
  </body>
</html>
"""

# Manual search using Python
if "<script>" in html_content:
    print("DANGER: Possible XSS vector found! <script> tag detected.")
else:
    print("Safe: No <script> tags found.")

DANGER: Possible XSS vector found! <script> tag detected.


### Your Turn: Automate the Scanner

Combine what we learned! Write a script below that:
1. Fetches a URL (ask the user for input or use a variable).
2. Checks if `Content-Security-Policy` is missing.
3. Checks if the text "\<script>" appears in the content (simplified check).

*(Note: Real-world scanners are much more complex, but this is the logic!)*

In [6]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 1. Target URL (change to any safe/permitted target)
target_url = "https://example.com"

try:
    response = requests.get(target_url, verify=False)

    # 2. Check for missing Content-Security-Policy header
    if "Content-Security-Policy" not in response.headers:
        print("[-] MISSING header: Content-Security-Policy — XSS protection may be weak!")
    else:
        print(f"[+] Found Content-Security-Policy: {response.headers['Content-Security-Policy']}")

    # 3. Check if <script> tag appears in the page body
    if "<script>" in response.text.lower():
        print("[!] WARNING: <script> tag found in page body — requires manual verification for XSS!")
    else:
        print("[+] No inline <script> tags detected in body.")

except Exception as e:
    print(f"[ERROR] Could not fetch URL: {e}")

[-] MISSING header: Content-Security-Policy — XSS protection may be weak!
[+] No inline <script> tags detected in body.


## Wrap-Up

Today we learned:
1. **Variables & Loops** allow us to automate repetitive tasks.
2. **Requests** lets Python talk to the web.
3. **Headers** reveal security configurations.
4. Small text patterns (like `<script>`) can indicate big vulnerabilities.

**Next Lesson:** We will dive into files, logs, and digital forensics!

In [7]:
# Solution for the "Your Turn" exercise
# Uncomment to see the solution

# url = input("Enter URL to scan: ")
# try:
#     r = requests.get(url)
#     if "Content-Security-Policy" not in r.headers:
#         print("[-] Missing Content-Security-Policy header")
#     if "<script>" in r.text:
#         print("[!] <script> tag found in body (requires manual verification)")
# except Exception as e:
#     print("Error fetching URL:", e)

---

##  Reflection — What I Learned Today

In this lesson, I was introduced to the foundational concepts of Python programming through a cybersecurity lens. Rather than learning syntax in isolation, every concept was applied directly to a real security task, which made the material both engaging and meaningful.

### Key Takeaways

**Python Fundamentals**
I learned how to store data in variables and work with strings and lists. The loop structure allowed me to automate repetitive tasks — for example, iterating over a list of paths to construct multiple URLs automatically. This immediately demonstrated why automation is central to both offensive and defensive security work.

**HTTP and the `requests` Library**
I discovered that Python can interact with the web just like a browser does, by sending HTTP GET requests and receiving responses. Understanding the structure of an HTTP response — status code, headers, and body — gave me a clearer mental model of how web communication works at a technical level.

**Security Headers**
I learned that HTTP response headers carry important security metadata. The absence of headers such as `Content-Security-Policy` or `X-Frame-Options` can expose a web application to attacks like XSS or Clickjacking. Checking for these headers programmatically is a standard step in web security assessments.

**Cross-Site Scripting (XSS)**
I was introduced to XSS as a class of vulnerability where unsanitized user input allows malicious scripts to execute in a victim's browser. By searching for `<script>` tags in HTML content, I performed a simplified but conceptually accurate automated detection.

**Building the Scanner**
By combining all the above concepts, I wrote an automated scanner that fetches a URL, checks for a missing security header, and flags potential XSS vectors. This exercise showed how a few lines of Python can replicate the core logic of professional security tools.

### Personal Reflection
What surprised me most was how quickly Python allows you to go from a concept to a working tool. The attacker/defender mindset introduced at the start of the lesson also shifted how I think about code: every input is potentially untrusted, and every missing configuration is a potential weakness. I feel confident applying these concepts to the next lesson.